# RoadVision AI - Dataset verification

## 1. Download the dataset

In [2]:
import kagglehub
from pathlib import Path

path = kagglehub.dataset_download("ashfakyeafi/road-vehicle-images-dataset")
print("Downloaded to:", path)

Using Colab cache for faster access to the 'road-vehicle-images-dataset' dataset.
Downloaded to: /kaggle/input/road-vehicle-images-dataset


## 2. Locate the dataset folder

In [3]:
ROOT = Path(path) / "trafic_data"
print("ROOT:", ROOT)
print("ROOT exists:", ROOT.exists())

print("\nContent of ROOT:")
for p in sorted(ROOT.iterdir()):
    print(("[DIR]  " if p.is_dir() else "[FILE] ") + p.name)

ROOT: /kaggle/input/road-vehicle-images-dataset/trafic_data
ROOT exists: True

Content of ROOT:
[FILE] data_1.yaml
[DIR]  train
[DIR]  valid


## 3. Locations of images and labels in each split

In [4]:
SPLITS = ["train", "valid", "test"]

for split in SPLITS:
    img_dir = ROOT / split / "images"
    lbl_dir = ROOT / split / "labels"
    print(f"[{split}]")
    print("  images:", img_dir, "->", "EXISTS" if img_dir.exists() else "MISSING")
    print("  labels:", lbl_dir, "->", "EXISTS" if lbl_dir.exists() else "MISSING")

[train]
  images: /kaggle/input/road-vehicle-images-dataset/trafic_data/train/images -> EXISTS
  labels: /kaggle/input/road-vehicle-images-dataset/trafic_data/train/labels -> EXISTS
[valid]
  images: /kaggle/input/road-vehicle-images-dataset/trafic_data/valid/images -> EXISTS
  labels: /kaggle/input/road-vehicle-images-dataset/trafic_data/valid/labels -> EXISTS
[test]
  images: /kaggle/input/road-vehicle-images-dataset/trafic_data/test/images -> MISSING
  labels: /kaggle/input/road-vehicle-images-dataset/trafic_data/test/labels -> MISSING


## 4. Count images and annotation files, check pairing

In [5]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

total_img, total_lbl = 0, 0

for split in SPLITS:
    img_dir = ROOT / split / "images"
    lbl_dir = ROOT / split / "labels"
    if not (img_dir.exists() and lbl_dir.exists()):
        print(f"[{split}] skipped (folder missing)")
        continue

    images = {p.stem for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXT}
    labels = {p.stem for p in lbl_dir.glob("*.txt")}

    no_label = sorted(images - labels)
    no_image = sorted(labels - images)

    total_img += len(images)
    total_lbl += len(labels)

    print(f"[{split}] images={len(images)} | labels={len(labels)} | "
          f"images without label={len(no_label)} | labels without image={len(no_image)}")
    if no_label:
        print("   e.g. image without label:", no_label[:3])
    if no_image:
        print("   e.g. label without image:", no_image[:3])

print(f"\nTOTAL images={total_img} | TOTAL labels={total_lbl}")

[train] images=2704 | labels=2704 | images without label=0 | labels without image=0
[valid] images=300 | labels=300 | images without label=0 | labels without image=0
[test] skipped (folder missing)

TOTAL images=3004 | TOTAL labels=3004


## 5. Inspect `data_1.yaml` and class mapping

In [6]:
import yaml

yaml_path = ROOT / "data_1.yaml"
print("YAML exists:", yaml_path.exists())

with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

print("train:", cfg["train"])
print("val  :", cfg["val"])
print("nc   :", cfg["nc"])
print("len(names):", len(cfg["names"]), "| nc == len(names):", cfg["nc"] == len(cfg["names"]))

print("\nClass ID -> class name")
for class_id, name in enumerate(cfg["names"]):
    print(f"{class_id:2d} -> {name}")

YAML exists: True
train: ../train/images
val  : ../valid/images
nc   : 21
len(names): 21 | nc == len(names): True

Class ID -> class name
 0 -> ambulance
 1 -> army vehicle
 2 -> auto rickshaw
 3 -> bicycle
 4 -> bus
 5 -> car
 6 -> garbagevan
 7 -> human hauler
 8 -> minibus
 9 -> minivan
10 -> motorbike
11 -> pickup
12 -> policecar
13 -> rickshaw
14 -> scooter
15 -> suv
16 -> taxi
17 -> three wheelers -CNG-
18 -> truck
19 -> van
20 -> wheelbarrow


## 6. Do the YAML paths match the real folders?
YAML paths are relative to the YAML file location.

In [7]:
for key in ["train", "val"]:
    resolved = (yaml_path.parent / cfg[key]).resolve()
    print(f"{key}: {cfg[key]} -> {resolved} -> {'EXISTS' if resolved.exists() else 'DOES NOT EXIST'}")

print("\nReal folders:")
print("train/images:", (ROOT / "train/images").exists())
print("valid/images:", (ROOT / "valid/images").exists())

train: ../train/images -> /kaggle/input/road-vehicle-images-dataset/train/images -> DOES NOT EXIST
val: ../valid/images -> /kaggle/input/road-vehicle-images-dataset/valid/images -> DOES NOT EXIST

Real folders:
train/images: True
valid/images: True


## 7. Check class IDs in labels match the YAML

In [8]:
from collections import Counter

nc = cfg["nc"]
class_count = Counter()
bad_lines = 0
empty_files = 0

for split in SPLITS:
    lbl_dir = ROOT / split / "labels"
    if not lbl_dir.exists():
        continue
    for f in lbl_dir.glob("*.txt"):
        lines = [l.split() for l in f.read_text().splitlines() if l.strip()]
        if not lines:
            empty_files += 1
        for l in lines:
            if len(l) != 5 or not (0 <= int(float(l[0])) < nc):
                bad_lines += 1
            else:
                class_count[int(l[0])] += 1

print("Empty label files (image with no vehicle):", empty_files)
print("Bad lines (wrong format or class id out of range):", bad_lines)
print("Max class id found:", max(class_count), "| allowed max:", nc - 1)

print("\nObjects per class:")
for i, name in enumerate(cfg["names"]):
    print(f"{i:2d} {name:22s} {class_count.get(i, 0)}")

Empty label files (image with no vehicle): 2
Bad lines (wrong format or class id out of range): 0
Max class id found: 20 | allowed max: 20

Objects per class:
 0 ambulance              70
 1 army vehicle           43
 2 auto rickshaw          372
 3 bicycle                459
 4 bus                    3339
 5 car                    5474
 6 garbagevan             3
 7 human hauler           169
 8 minibus                95
 9 minivan                932
10 motorbike              2284
11 pickup                 1225
12 policecar              32
13 rickshaw               3539
14 scooter                38
15 suv                    858
16 taxi                   60
17 three wheelers -CNG-   2986
18 truck                  1495
19 van                    755
20 wheelbarrow            120
